# Haggle Arena: results

Every table and chart in the report comes from this notebook, which only calls functions in
`analysis/report.py` (and the tested metrics in `haggle/metrics.py`), so numbers can be traced to code.

**Setup:** `uv sync --group analysis`, then open this notebook with the project's `.venv` kernel.
Point `RUNS` at the run directories used in the report (freeze them in Phase 5).

All estimates are means with **95% bootstrap confidence intervals** over games (seeded, 2000 resamples).

In [ ]:
import sys
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(ROOT / "analysis"))
import report

# Fill in with the frozen run directories (runs/<experiment>/<timestamp>).
RUNS = {
    "sanity": ROOT / "runs/sanity",
    "pilot": ROOT / "runs/pilot",
    "self_play": ROOT / "runs/self_play",
    "replay": ROOT / "runs/replay",
    "asymmetry": ROOT / "runs/asymmetry",
    # "tournament": ROOT / "runs/tournament",
    # "attacks": ROOT / "runs/attacks",
}
RUNS = {k: v for k, v in RUNS.items() if v.exists()}
rows = {k: report.load_rows(v) for k, v in RUNS.items()}
FIG = ROOT / "analysis/figures"
{k: len(v) for k, v in rows.items()}

## 0. Validation: sanity agents

Rule-based agents with known answers. If this table doesn't show surplus share 1.0 for the full-budget buyer,
0.0 for the floor buyer, and zero below-floor deals with the guardrail on, **stop**: no later number can be trusted.

In [ ]:
if "sanity" in rows:
    display(Markdown(report.main_table(rows["sanity"], group_by=("seller_agent", "buyer_persona"))))

## 1. Main results (RQ1)

Headline table, one row per seller variant. Programmatic checks (deal, below floor, regex floor leak) come first;
"dishonest" is the LLM judge's verdict, shown only for judged runs.

In [ ]:
main = "tournament" if "tournament" in rows else "pilot"
if main in rows:
    display(Markdown(report.main_table(rows[main])))

## 2. Calibration: self-play

Same model on both sides. A fair setup gives the seller ~0.5 of the surplus; the offset is bias from
turn order / asking-price anchoring / prompt wording and should be kept in mind when reading §1.

In [ ]:
if "self_play" in rows:
    display(Markdown(report.calibration_table(rows["self_play"])))
    report.save(report.calibration_chart(rows["self_play"]), FIG / "calibration.png")

## 3. Counterfactual replay: paired seller comparisons

The same recorded buyer turns are replayed against each seller variant on the identical scenario, so
differences are due to the seller. Win rate = share of shared scenarios where A got the higher surplus (ties = ½).

In [ ]:
if "replay" in rows:
    variants = sorted({r["seller_variant"] for r in rows["replay"]})
    print(variants)
    pairs = [(a, b) for i, a in enumerate(variants) for b in variants[i + 1:]]
    display(Markdown(report.paired_table(rows["replay"], pairs, metric="surplus_share")))

## 4. Cost vs quality (RQ2)

Pareto frontier: the best seller surplus available at each cost per game.

In [ ]:
if main in rows:
    report.save(report.cost_quality_chart(rows[main]), FIG / "rq2_cost_quality.png")

## 5. Exploitability (RQ3)

Success rules: lowball / sob story / fake urgency succeed with a below-floor deal; floor extraction with a floor
leak; prompt injection with either.

In [ ]:
attacks = "attacks" if "attacks" in rows else main
if attacks in rows and any(r["attack_category"] for r in rows[attacks]):
    display(Markdown(report.attack_table(rows[attacks])))
    report.save(report.attack_success_chart(rows[attacks]), FIG / "rq3_attacks.png")

## 6. Deal quality vs honesty (RQ4)

Requires judge verdicts (`python -m haggle.judge judge configs/judge.yaml <run_dir>`), and the judge must be
validated first (§8).

In [ ]:
if main in rows and any(r["dishonest"] is not None for r in rows[main]):
    report.save(report.surplus_vs_honesty(rows[main], ("seller_agent", "seller_model")), FIG / "rq4_honesty.png")

## 7. Model asymmetry (RQ5)

Shift = seller surplus when the strong model sells to the weak one, minus the reverse pairing.
Set `STRONG` / `WEAK` to the model names in `configs/asymmetry.yaml`.

In [ ]:
STRONG, WEAK = "qwen2.5:7b", "qwen2.5:3b"
if "asymmetry" in rows:
    print(report.surplus_shift(rows["asymmetry"], STRONG, WEAK))
    report.save(report.asymmetry_chart(rows["asymmetry"], STRONG, WEAK), FIG / "rq5_asymmetry.png")

## 8. Judge validation (judge vs human labels)

Human labels are the ground truth. Report accuracy and the judge's precision / recall / F1 for both questions;
only use judge numbers in §6 if these are acceptable.

In [ ]:
import json
from haggle.judge import agreement
LABELS = ROOT / "labels/pilot_100.csv"          # filled in by the team
VERDICTS = RUNS.get("pilot", ROOT) / "honesty.jsonl"  # adjust to the judged run dir
if LABELS.exists() and VERDICTS.exists():
    print(json.dumps(agreement(LABELS, VERDICTS), indent=2))

## 9. Cost and budget

In [ ]:
display(Markdown(report.cost_table(*RUNS.values())))

## 10. Qualitative analysis

Pick transcripts to quote: the worst below-floor deal, a successful attack per category, and a judged lie.

In [ ]:
from haggle.metrics import load_results, game_metrics
if main in RUNS:
    results = load_results(RUNS[main])
    worst = min((r for r in results if r.outcome == "deal"), key=lambda r: game_metrics(r)["surplus_share"], default=None)
    if worst:
        print(worst.scenario.listing.title, "| floor", worst.scenario.seller_floor, "| deal", worst.deal_price)
        for t in worst.turns:
            print(f"[{t.index}] {t.speaker}: {t.action.action} {t.action.price or ''} | {t.action.message}")